# **PREPROCESAMIENTO DEL DATASET: GÉNEROS Y ARTISTAS**

In [2]:
import pandas as pd
import numpy as np
import re
import ast
import html
from pathlib import Path

In [3]:

# 1. Cargar el documento CSV
ruta_archivo = 'datasets scraping/dataset_fusionado.csv'

try:
    df = pd.read_csv(ruta_archivo)
    
    # 2. Obtener el número de filas y columnas usando .shape
    num_filas, num_columnas = df.shape
    print("=== INFORMACIÓN GENERAL ===")
    print(f"Número de filas: {num_filas}")
    print(f"Número de columnas: {num_columnas}\n")
    
    # 3. Mostrar los nombres de las columnas usando .columns
    print("=== NOMBRES DE LAS COLUMNAS ===")
    for columna in df.columns:
        print(f"- {columna}")
    print("\n")
    
    # 4. Ver cuántas filas están vacías (nulas) por columna usando .isnull().sum()
    print("=== VALORES VACÍOS POR COLUMNA ===")
    valores_vacios = df.isnull().sum()
    print(valores_vacios)

except FileNotFoundError:
    print(f"Error: No se encontró el archivo '{ruta_archivo}'. Verifica que el nombre esté escrito correctamente y esté en la misma carpeta que este script.")
except Exception as e:
    print(f"Ocurrió un error inesperado al leer el archivo: {e}")

=== INFORMACIÓN GENERAL ===
Número de filas: 10081
Número de columnas: 8

=== NOMBRES DE LAS COLUMNAS ===
- artist
- song
- lyrics
- genres
- genre_1
- genre_2
- genre_3
- <<<<<<< HEAD


=== VALORES VACÍOS POR COLUMNA ===
artist          3715
song            3715
lyrics          4814
genres          4449
genre_1         4449
genre_2         4466
genre_3         4467
<<<<<<< HEAD    9166
dtype: int64


In [4]:
# cargar dataset
DATA_PATH = "datasets scraping/dataset_fusionado.csv"

df = pd.read_csv(DATA_PATH)

print("Shape inicial:", df.shape)
display(df.head())
display(df.tail())

Shape inicial: (10081, 8)


,artist,song,lyrics,genres,genre_1,genre_2,genre_3,<<<<<<< HEAD
0,/moonspell/,Heartshaped Abyss,Never resist\r\nThe firewalker\r\nHe has been ...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
1,/moonspell/,Let The Children Cum To Me...,"Hey you little Jesus' bride, why have you smil...","gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
2,/moonspell/,Memento Mori,"I am the moment, the soul\r\nThe moment that e...","gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
3,/moonspell/,Once It Was Ours!,Vanishing act inside the weak\r\nIn need of yo...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
4,/moonspell/,Serpent Angel,Father Satan send the Serpent\r\nPoison me wit...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN


,artist,song,lyrics,genres,genre_1,genre_2,genre_3,<<<<<<< HEAD
10076,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10077,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10078,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10079,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10080,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Eliminar la columna HEAD
df = df.drop(columns=["<<<<<<< HEAD"], errors="ignore")

# Normalizar columnas
df.columns = df.columns.str.strip().str.lower()

expected_cols = ["artist", "song", "lyrics", "genres", "genre_1", "genre_2", "genre_3"]

missing_cols = [col for col in expected_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas en el dataset: {missing_cols}")

print("Columnas disponibles:")
print(df.columns.tolist())

Columnas disponibles:
['artist', 'song', 'lyrics', 'genres', 'genre_1', 'genre_2', 'genre_3']


# **PREPROCESADO DE ARTIST**

In [6]:

def clean_artist(text):
    """
    Limpieza ligera para nombres de artistas.
    Elimina barras (/), decodifica HTML y limpia espacios.
    """
    if pd.isna(text):
        return np.nan
    
    text = str(text)
    
    # Decodificar entidades HTML (ej: Beyonc&eacute; -> Beyoncé)
    text = html.unescape(text)
    
    # Eliminar URLs (por si el scraper capturó un enlace por error)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    
    # Normalizar comillas y apóstrofes
    text = (
        text.replace("“", '"')
            .replace("”", '"')
            .replace("‘", "'")
            .replace("’", "'")
    )
    
    # Eliminar las barras "/" (reemplazándolas por un espacio para evitar fusionar palabras)
    text = text.replace("/", " ")
    
    # Normalizar saltos de línea y tabs
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    
    # Eliminar espacios múltiples y recortar los extremos
    text = re.sub(r"\s+", " ", text).strip()
    
    if text == "":
        return np.nan
    
    return text

# Aplicar la limpieza a la columna artist
df["artist_clean"] = df["artist"].apply(clean_artist)

# Mostrar una pequeña comprobación de los resultados
display(df[["artist", "artist_clean", "song"]].head(10))

,artist,artist_clean,song
0,/moonspell/,moonspell,Heartshaped Abyss
1,/moonspell/,moonspell,Let The Children Cum To Me...
2,/moonspell/,moonspell,Memento Mori
3,/moonspell/,moonspell,Once It Was Ours!
4,/moonspell/,moonspell,Serpent Angel
5,/moonspell/,moonspell,The Darkening
6,/moonspell/,moonspell,Upon The Blood Of Men
7,/running-wild/,running-wild,White Buffalo
8,/running-wild/,running-wild,Billy The Kid
9,/running-wild/,running-wild,Branded And Exiled


# **PREPROCESADO DE GENRES**

In [7]:

# 1. Combinar las tres columnas de géneros en una sola serie (lista larga)
todos_los_generos = pd.concat([df['genre_1'], df['genre_2'], df['genre_3']])

# 2. Eliminar valores nulos (NaN) y obtener los valores únicos
generos_unicos = todos_los_generos.dropna().unique()

# 3. Limpiar espacios extra y poner en minúsculas para evitar duplicados (ej: "Pop" y " pop ")
# Convertimos a serie temporalmente para aplicar métodos de texto de pandas
generos_limpios = pd.Series(generos_unicos).str.strip().str.lower().unique()

# 4. (Opcional) Ordenar la lista alfabéticamente para que sea más fácil de leer
generos_limpios = sorted(generos_limpios)

print("=== GÉNEROS ÚNICOS EN EL DATASET ===")
print(f"Total de géneros distintos: {len(generos_limpios)}\n")
print(generos_limpios)

=== GÉNEROS ÚNICOS EN EL DATASET ===
Total de géneros distintos: 1258

['-death metal-', '00s', '1 star', '10 of 10 stars', '10 stars', '10s', '123 drippy sappy', '128 bpm', '1947', '1957', '1959', '1960s', '1962', '1965', '1967', '1969', '1970', '1971', '1973', '1976', '1977', '1978', '1979', '1980s', '1981', '1983', '1984', '1987', '1989', '1990', '1990s', '1991', '1992', '1993', '1994', '1996', '1997', '1998', '1999', '2 of 10 stars', '2000s', '2001', '2002', '2003', '2005', '2006', '2007', '2008', '2009', '2010', '2010s', '2012', '2013', '2013 single', '2014', '2015', '2015 gif', '2016', '2017', '2018 single', '2019', '2020', '2020s', '2021', '2022', '20th century', '2ne1', '3-star', '3l3ktr0 ch1x', '4 of 10 stars', '40', '40s', '5 of 10 stars', "50's", '50s', '50s rock n roll', '6 of 10 stars', "60's", '600th play', '60s', '7 of 10 stars', '70s', '70s blockbusters', '77davez-all-tracks', '8 of 10 stars', '80', '80 bpm', '80s', '80s hard rock', '80s power metal', '80s rhythm and bl

In [24]:
# ============================================================
# Agrupación automática de géneros con Transformer Zero-Shot
# ============================================================

import pandas as pd
import numpy as np
import re
import ast
import html
from tqdm.auto import tqdm
from transformers import pipeline

# ============================================================
# 1. Macrogéneros objetivo
# ============================================================

MACRO_GENRES = [
    "Rock",
    "Pop",
    "Country",
    "Electronic / Dance",
    "Folk / Traditional",
    "R&B / Soul / Funk",
    "Punk / Emo / Hardcore",
    "Metal",
    "Hip Hop / Rap",
    "Experimental",
    "Jazz",
    "Blues",
    "Latin",
    "Reggae / Caribbean",
    "Classical / Art Music",
    "Other"
]

# ============================================================
# 2. Modelo zero-shot
# ============================================================

ZERO_SHOT_MODEL = "facebook/bart-large-mnli"

genre_classifier = pipeline(
    "zero-shot-classification",
    model=ZERO_SHOT_MODEL
)

# ============================================================
# 3. Limpieza mínima de etiquetas de género
# ============================================================

def normalize_raw_genre(genre):
    """
    Normaliza mínimamente una etiqueta de género.
    No valida contra listas manuales.
    """
    if pd.isna(genre):
        return None

    genre = str(genre)
    genre = html.unescape(genre)
    genre = genre.strip().lower()
    genre = genre.strip("'").strip('"')
    genre = genre.replace("_", " ")
    genre = re.sub(r"\s+", " ", genre).strip()
    genre = genre.strip(" .;:,")

    if genre in ["", "nan", "none", "null", "unknown", "[]", "-", "n/a", "na"]:
        return None

    return genre


def parse_raw_genre_cell(value):
    """
    Convierte una celda de género en una lista de géneros raw normalizados.
    Soporta:
    - valor simple: "pop"
    - lista string: "['pop', 'rock']"
    - valores separados por comas: "pop, rock, r&b"
    """
    if pd.isna(value):
        return []

    value = str(value).strip()
    parsed_values = []

    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, (list, tuple, set)):
                parsed_values = list(parsed)
            else:
                parsed_values = [value]

        except Exception:
            parsed_values = [value]

    elif "," in value:
        parsed_values = value.split(",")

    else:
        parsed_values = [value]

    normalized = []

    for genre in parsed_values:
        genre_norm = normalize_raw_genre(genre)

        if genre_norm is not None:
            normalized.append(genre_norm)

    # Eliminar duplicados manteniendo orden
    unique_genres = []
    seen = set()

    for genre in normalized:
        if genre not in seen:
            unique_genres.append(genre)
            seen.add(genre)

    return unique_genres

# ============================================================
# 4. Extraer todos los géneros únicos del dataset
# ============================================================

genre_cols = ["genres", "genre_1", "genre_2", "genre_3"]

all_raw_genres = set()

for col in genre_cols:
    if col in df.columns:
        for value in df[col].dropna():
            genres = parse_raw_genre_cell(value)
            all_raw_genres.update(genres)

all_raw_genres = sorted(all_raw_genres)

print("Número de géneros/subgéneros únicos detectados:", len(all_raw_genres))
print(all_raw_genres[:50])

# ============================================================
# 5. Clasificar automáticamente cada subgénero en macrogénero
# ============================================================

def classify_genre_zero_shot(genre, classifier, candidate_labels, min_confidence=0.30):
    """
    Clasifica un género/subgénero en un macrogénero usando zero-shot classification.
    Si la confianza es baja, lo asigna a 'Other'.
    """

    if genre is None or pd.isna(genre):
        return {
            "raw_genre": genre,
            "predicted_macro_genre": None,
            "confidence": np.nan,
            "all_labels": None,
            "all_scores": None
        }

    text = f"The music genre is {genre}."

    result = classifier(
        sequences=text,
        candidate_labels=candidate_labels,
        hypothesis_template="This music genre belongs to {}.",
        multi_label=False
    )

    top_label = result["labels"][0]
    top_score = result["scores"][0]

    if top_score < min_confidence:
        top_label = "Other"

    return {
        "raw_genre": genre,
        "predicted_macro_genre": top_label,
        "confidence": float(top_score),
        "all_labels": result["labels"],
        "all_scores": result["scores"]
    }


genre_classification_results = []

for genre in tqdm(all_raw_genres, desc="Clasificando géneros con Transformer"):
    result = classify_genre_zero_shot(
        genre=genre,
        classifier=genre_classifier,
        candidate_labels=MACRO_GENRES,
        min_confidence=0.30
    )

    genre_classification_results.append(result)

df_genre_transformer_map = pd.DataFrame(genre_classification_results)

display(
    df_genre_transformer_map
    .sort_values("confidence", ascending=True)
    .head(30)
)

# ============================================================
# 6. Crear diccionario automático subgénero -> macrogénero
# ============================================================

TRANSFORMER_GENRE_TO_GROUP = dict(
    zip(
        df_genre_transformer_map["raw_genre"],
        df_genre_transformer_map["predicted_macro_genre"]
    )
)

display(
    df_genre_transformer_map[
        ["raw_genre", "predicted_macro_genre", "confidence"]
    ].sort_values("raw_genre")
)


# ============================================================
# 7. Parsear celda y devolver macrogéneros automáticos
# ============================================================

def parse_genre_cell_transformer(value):
    """
    Convierte una celda de género en una lista de macrogéneros usando
    el mapeo automático generado por el Transformer.
    """

    raw_genres = parse_raw_genre_cell(value)

    macro_genres = []

    for genre in raw_genres:
        macro = TRANSFORMER_GENRE_TO_GROUP.get(genre)

        if macro is not None and pd.notna(macro):
            macro_genres.append(macro)

    # Eliminar duplicados manteniendo orden
    unique_macro_genres = []
    seen = set()

    for macro in macro_genres:
        if macro not in seen:
            unique_macro_genres.append(macro)
            seen.add(macro)

    return unique_macro_genres


# ============================================================
# 8. Aplicar a genre_1, genre_2 y genre_3
# ============================================================

genre_cols = ["genre_1", "genre_2", "genre_3"]

for col in genre_cols:
    df[col + "_clean"] = df[col].apply(
        lambda x: parse_genre_cell_transformer(x)[0]
        if len(parse_genre_cell_transformer(x)) > 0
        else None
    )

# ============================================================
# 9. Construir lista final de macrogéneros por canción
# ============================================================

def build_genre_list(row):
    """
    Construye una lista de macrogéneros únicos por canción.
    """

    genres = []

    for col in ["genre_1_clean", "genre_2_clean", "genre_3_clean"]:
        genre = row[col]

        if genre is not None and pd.notna(genre):
            genres.append(genre)

    unique_genres = []
    seen = set()

    for genre in genres:
        if genre not in seen:
            unique_genres.append(genre)
            seen.add(genre)

    return unique_genres


df["genre_list"] = df.apply(build_genre_list, axis=1)
df["n_genres"] = df["genre_list"].apply(len)

# ============================================================
# 10. Comprobaciones
# ============================================================

print("Ejemplo parse_genre_cell_transformer('alternative rock'):")
print(parse_genre_cell_transformer("alternative rock"))

print("\nEjemplo parse_genre_cell_transformer('pop, r&b, trap'):")
print(parse_genre_cell_transformer("pop, r&b, trap"))

print("\nNúmero de canciones sin macrogénero asignado:")
print((df["n_genres"] == 0).sum())

print("\nNúmero de canciones con al menos un macrogénero:")
print((df["n_genres"] > 0).sum())

print("\nDistribución de macrogéneros:")
display(
    df.explode("genre_list")["genre_list"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "genre_group", "genre_list": "n_songs"})
)

display(
    df[
        [
            "artist",
            "song",
            "genre_1",
            "genre_2",
            "genre_3",
            "genre_1_clean",
            "genre_2_clean",
            "genre_3_clean",
            "genre_list",
            "n_genres"
        ]
    ].head()
)




Loading weights: 100%|██████████| 515/515 [00:00<00:00, 6422.00it/s]


Número de géneros/subgéneros únicos detectados: 1259
['-death metal-', '00s', '1 star', '10 of 10 stars', '10 stars', '10s', '123 drippy sappy', '128 bpm', '1947', '1957', '1959', '1960s', '1962', '1965', '1967', '1969', '1970', '1971', '1973', '1976', '1977', '1978', '1979', '1980s', '1981', '1983', '1984', '1987', '1989', '1990', '1990s', '1991', '1992', '1993', '1994', '1996', '1997', '1998', '1999', '2 of 10 stars', '2000s', '2001', '2002', '2003', '2005', '2006', '2007', '2008', '2009', '2010']


Clasificando géneros con Transformer: 100%|██████████| 1259/1259 [10:00<00:00,  2.10it/s]


,raw_genre,predicted_macro_genre,confidence,all_labels,all_scores
334,contemporary christian,Other,0.094803,"[Pop, Other, Country, Experimental, Latin, Roc...","[0.09480293840169907, 0.09391624480485916, 0.0..."
789,madonna,Other,0.096673,"[Other, Pop, Latin, Rock, Metal, Experimental,...","[0.09667307138442993, 0.09360723197460175, 0.0..."
297,christian,Other,0.096676,"[Other, Folk / Traditional, Pop, Experimental,...","[0.09667590260505676, 0.09608479589223862, 0.0..."
98,a perfect ending to a perfect album,Other,0.098636,"[Classical / Art Music, Folk / Traditional, El...","[0.09863566607236862, 0.09245239943265915, 0.0..."
497,featuring,Other,0.099414,"[Folk / Traditional, Classical / Art Music, El...","[0.09941423684358597, 0.0954611673951149, 0.08..."
1151,the wiz,Other,0.100437,"[Rock, Jazz, Electronic / Dance, Hip Hop / Rap...","[0.10043749958276749, 0.0998217985033989, 0.09..."
650,immediate,Other,0.101935,"[Electronic / Dance, Folk / Traditional, Regga...","[0.10193504393100739, 0.08841016888618469, 0.0..."
845,my favorite,Other,0.103350,"[Classical / Art Music, Folk / Traditional, El...","[0.10334968566894531, 0.10196343064308167, 0.0..."
7,128 bpm,Other,0.103789,"[Folk / Traditional, Electronic / Dance, Class...","[0.10378856956958771, 0.10260611027479172, 0.0..."
1114,superfavourits,Other,0.103803,"[Folk / Traditional, Classical / Art Music, El...","[0.1038033738732338, 0.09160511195659637, 0.08..."


,raw_genre,predicted_macro_genre,confidence
0,-death metal-,Metal,0.945796
1,00s,Other,0.177716
2,1 star,Other,0.110153
3,10 of 10 stars,Other,0.117671
4,10 stars,Other,0.107610
...,...,...,...
1254,youthful praise,Other,0.159890
1255,yule,Folk / Traditional,0.435227
1256,zac brown band,Other,0.171183
1257,zajebisty bit,Other,0.203981


Ejemplo parse_genre_cell_transformer('alternative rock'):
['Rock']

Ejemplo parse_genre_cell_transformer('pop, r&b, trap'):
['Pop', 'R&B / Soul / Funk', 'Other']

Número de canciones sin macrogénero asignado:
4449

Número de canciones con al menos un macrogénero:
5632

Distribución de macrogéneros:


,n_songs,count
0,Other,3854
1,Rock,1070
2,Pop,1039
3,Metal,1012
4,Folk / Traditional,969
5,Country,878
6,Hip Hop / Rap,873
7,Electronic / Dance,857
8,R&B / Soul / Funk,148
9,Latin,145


,artist,song,genre_1,genre_2,genre_3,genre_1_clean,genre_2_clean,genre_3_clean,genre_list,n_genres
0,/moonspell/,Heartshaped Abyss,gothic metal,doom metal,dark metal,Metal,Metal,Metal,[Metal],1
1,/moonspell/,Let The Children Cum To Me...,gothic metal,doom metal,dark metal,Metal,Metal,Metal,[Metal],1
2,/moonspell/,Memento Mori,gothic metal,doom metal,dark metal,Metal,Metal,Metal,[Metal],1
3,/moonspell/,Once It Was Ours!,gothic metal,doom metal,dark metal,Metal,Metal,Metal,[Metal],1
4,/moonspell/,Serpent Angel,gothic metal,doom metal,dark metal,Metal,Metal,Metal,[Metal],1


Se utiliza el modelo Transformer `facebook/bart-large-mnli` mediante clasificación *zero-shot*. Este modelo no requiere entrenamiento específico con los géneros del dataset, sino que compara cada etiqueta con una serie de macrogéneros candidatos. Para ello, una etiqueta como `alternative rock` se transforma en una frase del tipo `The music genre is alternative rock` y se evalúa frente a hipótesis como `This music genre belongs to Rock`, `Pop`, `Metal`, etc.

Primero se definen los macrogéneros finales del proyecto. Después se limpian las etiquetas originales eliminando nulos, espacios, comillas y formatos inconsistentes. A continuación, se extraen todos los géneros únicos del dataset y se clasifican automáticamente con el modelo. Para cada género se guarda el macrogénero más probable y su nivel de confianza. Si la confianza es inferior a `0.30`, se asigna a `Other`.

Finalmente, se crea un diccionario automático `subgénero → macrogénero`, que se aplica a `genre_1`, `genre_2` y `genre_3` para generar las columnas limpias y la lista final de macrogéneros por canción.


In [26]:
# NORMALIZAR GENERO

genre_cols = ["genre_1", "genre_2", "genre_3"]

def get_first_transformer_macro_genre(value):
    """
    Devuelve el primer macrogénero asignado por el modelo Transformer
    para una celda de género.
    """
    parsed_genres = parse_genre_cell_transformer(value)

    if len(parsed_genres) > 0:
        return parsed_genres[0]

    return None


for col in genre_cols:
    df[col + "_clean"] = df[col].apply(get_first_transformer_macro_genre)

display(
    df[
        [
            "genres",
            "genre_1",
            "genre_2",
            "genre_3",
            "genre_1_clean",
            "genre_2_clean",
            "genre_3_clean"
        ]
    ].head()
)

,genres,genre_1,genre_2,genre_3,genre_1_clean,genre_2_clean,genre_3_clean
0,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,Metal,Metal,Metal
1,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,Metal,Metal,Metal
2,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,Metal,Metal,Metal
3,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,Metal,Metal,Metal
4,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,Metal,Metal,Metal


In [27]:
# ============================================================
# CONSTRUIR LISTA FINAL DE MACROGÉNEROS POR CANCIÓN
# ============================================================

def build_genre_list(row):
    genres = []

    for col in ["genre_1_clean", "genre_2_clean", "genre_3_clean"]:
        genre = row[col]

        if genre is not None and pd.notna(genre):
            genres.append(genre)

    # Eliminar duplicados manteniendo el orden
    unique_genres = []
    seen = set()

    for genre in genres:
        if genre not in seen:
            unique_genres.append(genre)
            seen.add(genre)

    return unique_genres


df["genre_list"] = df.apply(build_genre_list, axis=1)
df["n_genres"] = df["genre_list"].apply(len)

display(
    df[
        [
            "artist",
            "song",
            "genre_1_clean",
            "genre_2_clean",
            "genre_3_clean",
            "genre_list",
            "n_genres"
        ]
    ].head()
)

,artist,song,genre_1_clean,genre_2_clean,genre_3_clean,genre_list,n_genres
0,/moonspell/,Heartshaped Abyss,Metal,Metal,Metal,[Metal],1
1,/moonspell/,Let The Children Cum To Me...,Metal,Metal,Metal,[Metal],1
2,/moonspell/,Memento Mori,Metal,Metal,Metal,[Metal],1
3,/moonspell/,Once It Was Ours!,Metal,Metal,Metal,[Metal],1
4,/moonspell/,Serpent Angel,Metal,Metal,Metal,[Metal],1


In [28]:
# ============================================================
# CONTAR PALABRAS Y FILTRAR CANCIONES VÁLIDAS
# ============================================================

def count_words(text):
    if pd.isna(text):
        return 0
    return len(re.findall(r"\b[\w']+\b", str(text)))

# Usar lyrics_clean si existe; si no, usar lyrics
TEXT_COL = "lyrics_clean" if "lyrics_clean" in df.columns else "lyrics"

df["word_count"] = df[TEXT_COL].apply(count_words)


MIN_WORDS = 1

initial_rows = len(df)

mask_valid_lyrics = df[TEXT_COL].notna() & (df["word_count"] >= MIN_WORDS)
mask_valid_genres = df["n_genres"] > 0

df_clean = df[mask_valid_lyrics & mask_valid_genres].copy()

print("Filas iniciales:", initial_rows)
print("Filas tras limpieza:", len(df_clean))
print("Canciones eliminadas:", initial_rows - len(df_clean))

print("\nMotivos principales:")
print("Sin letra válida:", (~mask_valid_lyrics).sum())
print("Sin género válido:", (~mask_valid_genres).sum())

Filas iniciales: 10081
Filas tras limpieza: 4767
Canciones eliminadas: 5314

Motivos principales:
Sin letra válida: 4814
Sin género válido: 4449


In [29]:
# ============================================================
# NORMALIZAR ARTISTA Y CANCIÓN PARA DETECTAR DUPLICADOS
# ============================================================

def normalize_text_id(text):
    """
    Normaliza texto para detectar duplicados.
    """
    
    if pd.isna(text):
        return ""
    
    text = str(text).strip().lower()
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text)
    
    return text


df_clean["artist_norm"] = df_clean["artist"].apply(normalize_text_id)
df_clean["song_norm"] = df_clean["song"].apply(normalize_text_id)

duplicates = df_clean.duplicated(
    subset=["artist_norm", "song_norm"],
    keep=False
)

print("Duplicados:", duplicates.sum())

# Mantener la versión con más palabras
df_clean = (
    df_clean
    .sort_values("word_count", ascending=False)
    .drop_duplicates(subset=["artist_norm", "song_norm"], keep="first")
    .sort_index()
    .copy()
)

print("Shape tras eliminar duplicados:", df_clean.shape)

Duplicados: 116
Shape tras eliminar duplicados: (4709, 16)


In [30]:
# ============================================================
# DATASET EXPANDIDO POR MACROGÉNERO
# ============================================================

df_genres = df_clean.explode("genre_list").copy()

df_genres = df_genres.rename(columns={"genre_list": "genre"})

df_genres = df_genres[df_genres["genre"].notna()].copy()

# Evitar que una misma canción aparezca dos veces en el mismo macrogénero
df_genres = df_genres.drop_duplicates(
    subset=["artist_norm", "song_norm", "genre"],
    keep="first"
)

print("Shape df_clean, una fila por canción:", df_clean.shape)
print("Shape df_genres, una fila por canción-género:", df_genres.shape)

# Usar artist_clean si existe; si no, artist
ARTIST_COL = "artist_clean" if "artist_clean" in df_genres.columns else "artist"

display(
    df_genres[
        [ARTIST_COL, "song", "genre", TEXT_COL, "word_count"]
    ].head()
)

Shape df_clean, una fila por canción: (4709, 16)
Shape df_genres, una fila por canción-género: (9189, 16)


,artist_clean,song,genre,lyrics,word_count
0,moonspell,Heartshaped Abyss,Metal,Never resist\r\nThe firewalker\r\nHe has been ...,167
1,moonspell,Let The Children Cum To Me...,Metal,"Hey you little Jesus' bride, why have you smil...",197
2,moonspell,Memento Mori,Metal,"I am the moment, the soul\r\nThe moment that e...",219
3,moonspell,Once It Was Ours!,Metal,Vanishing act inside the weak\r\nIn need of yo...,174
4,moonspell,Serpent Angel,Metal,Father Satan send the Serpent\r\nPoison me wit...,152


In [31]:
# ============================================================
# DISTRIBUCIÓN DE MACROGÉNEROS
# ============================================================

genre_counts = df_genres["genre"].value_counts()

genre_counts_df = (
    genre_counts
    .reset_index()
)

genre_counts_df.columns = ["genre", "n_songs"]

display(genre_counts_df)

print("Número total de géneros únicos:", df_genres["genre"].nunique())
print("Número total de registros canción-género:", len(df_genres))


,genre,n_songs
0,Other,3162
1,Metal,979
2,Rock,936
3,Hip Hop / Rap,790
4,Pop,766
5,Folk / Traditional,762
6,Electronic / Dance,643
7,Country,616
8,R&B / Soul / Funk,133
9,Latin,125


Número total de géneros únicos: 16
Número total de registros canción-género: 9189


In [32]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

#df_clean.to_csv(OUTPUT_DIR / "songs_clean_by_song.csv", index=False)
df_genres.to_csv(OUTPUT_DIR / "songs_clean_by_genre_exploded_transformer.csv", index=False)

print("Archivos guardados:")
print(OUTPUT_DIR / "songs_clean_by_genre_exploded_transformer.csv")


Archivos guardados:
outputs/songs_clean_by_genre_exploded_transformer.csv


En primer lugar, se realizó una limpieza del dataset original con el objetivo de preparar los nombres de artistas y los géneros musicales para el análisis posterior. Se eliminaron canciones sin letra disponible, canciones con letras demasiado cortas y registros sin ningún género válido.

Además, se normalizaron las etiquetas de género presentes en las columnas genre_1, genre_2 y genre_3, convirtiéndolas a minúsculas, eliminando espacios innecesarios y unificando algunas variantes frecuentes. Dado que una canción puede pertenecer a varios géneros, se generó un segundo dataset en formato expandido, donde cada fila representa una combinación canción-género. Este formato permite analizar posteriormente la distribución emocional de las letras para cada género musical.

| Archivo                                      | Nivel                       | Uso principal                   |
| -------------------------------------------- | --------------------------- | ------------------------------- |
| `dataset_fusionado.csv`                    | Una fila por canción        | Visualización compacta     |
| `songs_clean_by_genre_exploded.csv`          | Una fila por canción-género | Análisis completo por género    |

